# KhushiDL - Heart Disease Detector

In [308]:
# Importing necessary modules
import tensorflow as tf
import pandas as pd
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt

## Data Preprocessing

In [290]:
# Loading the data
train_data = pd.read_csv('data/FinalDataSets/train.csv', header=None)
validation_data = pd.read_csv('data/FinalDataSets/validation.csv', header=None)
test_data = pd.read_csv('data/FinalDataSets/test.csv', header=None)

In [291]:
print("Shape: ", train_data.shape)
train_data.info()

Shape:  (644, 14)
<class 'pandas.DataFrame'>
RangeIndex: 644 entries, 0 to 643
Data columns (total 14 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   0       644 non-null    float64
 1   1       644 non-null    float64
 2   2       644 non-null    float64
 3   3       644 non-null    float64
 4   4       644 non-null    float64
 5   5       644 non-null    float64
 6   6       644 non-null    float64
 7   7       644 non-null    float64
 8   8       644 non-null    float64
 9   9       644 non-null    float64
 10  10      644 non-null    float64
 11  11      644 non-null    float64
 12  12      644 non-null    float64
 13  13      644 non-null    float64
dtypes: float64(14)
memory usage: 70.6 KB


In [292]:
# seperating params
train_x = train_data.iloc[:, [x for x in range(13)]]
validation_x = validation_data.iloc[:, [x for x in range(13)]]
test_x = test_data.iloc[:, [x for x in range(13)]]

In [293]:
print("Train Shape: ", train_x.shape)
print("Validation Shape: ", validation_x.shape)
print("Test Shape: ", test_x.shape)

Train Shape:  (644, 13)
Validation Shape:  (138, 13)
Test Shape:  (138, 13)


In [294]:
# sperating labels
train_y = train_data.iloc[: , [13]].copy()
validation_y = validation_data.iloc[: , [13]].copy()
test_y = test_data.iloc[: , [13]].copy()

In [295]:
print("Train Shape: ", train_y.shape)
print("Validation Shape: ", validation_y.shape)
print("Test Shape: ", test_y.shape)

Train Shape:  (644, 1)
Validation Shape:  (138, 1)
Test Shape:  (138, 1)


In [296]:
test_y.info()

<class 'pandas.DataFrame'>
RangeIndex: 138 entries, 0 to 137
Data columns (total 1 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   13      138 non-null    float64
dtypes: float64(1)
memory usage: 1.2 KB


In [297]:
# Using standar scaler
scaler = StandardScaler()

train_x = scaler.fit_transform(train_x)

validation_x = scaler.transform(validation_x)
test_x = scaler.transform(test_x)

In [298]:
# Clipping 1,2,3,4 into 1 single class (to improve model accuracy)
train_y = train_y.replace(2, 1)
train_y = train_y.replace(3, 1)
train_y = train_y.replace(4, 1)

validation_y = validation_y.replace(2, 1)
validation_y = validation_y.replace(3, 1)
validation_y = validation_y.replace(4, 1)

test_y = test_y.replace(2, 1)
test_y = test_y.replace(3, 1)
test_y = test_y.replace(4, 1)

In [299]:
# testing data
print(train_y.value_counts())
print(validation_y.value_counts())
print(test_y.value_counts())

13 
1.0    360
0.0    284
Name: count, dtype: int64
13 
1.0    84
0.0    54
Name: count, dtype: int64
13 
0.0    73
1.0    65
Name: count, dtype: int64


### Data Pipeline

In [300]:
BATCH = 64
AUTOTUNE = tf.data.AUTOTUNE


train_ds = (
    tf.data.Dataset.from_tensor_slices((train_x, train_y))
    .batch(BATCH)
    .shuffle(buffer_size=train_x.shape[0])
    .prefetch(AUTOTUNE)
)

validation_ds = (
    tf.data.Dataset.from_tensor_slices((validation_x, validation_y))
    .batch(BATCH)
    .prefetch(AUTOTUNE)
)

test_ds = (
    tf.data.Dataset.from_tensor_slices((test_x, test_y))
    .batch(BATCH)
    .prefetch(AUTOTUNE)
)

## Model Architecture

In [301]:
# creating the actual model
model = tf.keras.Sequential([
    tf.keras.Input(shape=(13,)),
    
    # tf.keras.layers.Dense(100, activation='relu'),
    tf.keras.layers.Dense(50, activation='relu'),
    tf.keras.layers.Dense(15, activation='relu'),
    tf.keras.layers.Dense(1, activation='sigmoid')
])

model.summary()

Model: "sequential_16"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_48 (Dense)                │ (None, 50)             │           700 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_49 (Dense)                │ (None, 15)             │           765 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_50 (Dense)                │ (None, 1)              │            16 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,481 (5.79 KB)

 Trainable params: 1,481 (5.79 KB)

 Non-trainable params: 0 (0.00 B)

### Compiling and Training the Model

In [302]:
# Compiling the model
model.compile(
    optimizer='adam',
    # loss='sparse_categorical_crossentropy',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

In [303]:
# EarlyStopping callback
earlyStopper = tf.keras.callbacks.EarlyStopping(
    monitor='val_loss',
    patience=5,
    restore_best_weights=True
)

In [304]:
# LR on Plateau callback
lr_callback = tf.keras.callbacks.ReduceLROnPlateau(
    monitor='val_loss',
    factor=.8,
    patience=2
)

In [305]:
# training the model
model_history = model.fit(
    train_ds,
    validation_data = validation_ds,
    epochs=32,
    callbacks=[earlyStopper]
)

Epoch 1/32


/Users/luckypawar/Data/Personal/Coding/AI/Learning/Projects/Heart Disease/venv/lib/python3.12/site-packages/keras/src/trainers/epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(


11/11 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.6755 - loss: 0.6350 - val_accuracy: 0.7754 - val_loss: 0.5704
Epoch 2/32
11/11 ━━━━━━━━━━━━━━━━━━━━ 0s 561us/step - accuracy: 0.7547 - loss: 0.5852 - val_accuracy: 0.8261 - val_loss: 0.5335
Epoch 3/32
11/11 ━━━━━━━━━━━━━━━━━━━━ 0s 556us/step - accuracy: 0.7811 - loss: 0.5454 - val_accuracy: 0.8333 - val_loss: 0.4987
Epoch 4/32
11/11 ━━━━━━━━━━━━━━━━━━━━ 0s 485us/step - accuracy: 0.7904 - loss: 0.5108 - val_accuracy: 0.8188 - val_loss: 0.4678
Epoch 5/32
11/11 ━━━━━━━━━━━━━━━━━━━━ 0s 488us/step - accuracy: 0.8012 - loss: 0.4809 - val_accuracy: 0.8043 - val_loss: 0.4499
Epoch 6/32
11/11 ━━━━━━━━━━━━━━━━━━━━ 0s 514us/step - accuracy: 0.8106 - loss: 0.4612 - val_accuracy: 0.8043 - val_loss: 0.4413
Epoch 7/32
11/11 ━━━━━━━━━━━━━━━━━━━━ 0s 478us/step - accuracy: 0.8137 - loss: 0.4475 - val_accuracy: 0.8043 - val_loss: 0.4365
Epoch 8/32
11/11 ━━━━━━━━━━━━━━━━━━━━ 0s 476us/step - accuracy: 0.8121 - loss: 0.4357 - val_accuracy: 0.8043 - val_lo

In [306]:
model.evaluate(test_ds)

3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 926us/step - accuracy: 0.8043 - loss: 0.4219


[0.42191627621650696, 0.804347813129425]

## Graphs & Saving the Model

In [307]:
# Saving the model
model.save('models/model_v1.keras')